# Feed Analyzer: train the models

Fine-tunes ModernBERT, Twitter-RoBERTa and DistilBERT on `model/data/processed/`
and saves each model to your Google Drive.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.
Each model takes about 5–15 minutes. When you're done, Runtime → Disconnect to save GPU quota.

In [ ]:
# 1. Check that a GPU is available (should print "Tesla T4").
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Connect Google Drive, where the trained models are saved.
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/feed_analyzer/models"

In [ ]:
# 3. Get the code. Set REPO_URL to your GitHub repo.
#    Private repo: you'll be asked for a GitHub token (Settings -> Developer settings
#    -> Personal access tokens, with read access to the repo).
REPO_URL = "https://github.com/sebastianp21/feed-analyzer.git"
BRANCH = "phase_2"  # change to "main" once phase 2 is merged
PRIVATE_REPO = False

import getpass, os
url = REPO_URL
if PRIVATE_REPO:
    token = getpass.getpass("GitHub token: ")
    url = REPO_URL.replace("https://", f"https://{token}@")
!rm -rf /content/feed_analyzer
!git clone -q --branch {BRANCH} {url} /content/feed_analyzer
%cd /content/feed_analyzer/model
!ls data/processed

In [ ]:
# 4. Install dependencies (Colab already has torch).
!pip install -q -r requirements.txt

In [ ]:
# 5. Train. Remove names from MODELS to train fewer.
#    Models are trained to local disk first (faster), then copied to Drive.
MODELS = ["modernbert", "roberta", "distilbert"]

for name in MODELS:
    print(f"\n========== {name} ==========")
    !python train.py --model {name} --out /content/output/{name}
    !mkdir -p {SAVE_DIR} && rm -rf {SAVE_DIR}/{name} && cp -r /content/output/{name} {SAVE_DIR}/{name}

In [ ]:
# 6. Compare validation scores. (The real comparison on the test split is Phase 3.)
import json, pandas as pd
rows = []
for name in MODELS:
    m = json.load(open(f"{SAVE_DIR}/{name}/val_metrics.json"))
    rows.append({"model": name, "accuracy": m["eval_accuracy"], "macro_f1": m["eval_macro_f1"]})
pd.DataFrame(rows).round(3)

## Use a model in the app

1. In Google Drive, download `feed_analyzer/models/modernbert` (Drive zips the folder).
2. Unzip it into `model/output/modernbert` in the repo on your computer.
3. Start the server with it:
   ```
   cd server
   MODEL_PATH=../model/output/modernbert uvicorn app.main:app --port 8000
   ```